In [1]:
# Resolve the CDR dataset and the workspace bucket
cdr    <- trimws(tail(system("wb resource resolve --id=C2024Q3R9", intern = TRUE), 1))
bucket <- trimws(tail(system("wb resource resolve --id=rw-migration-aou-rw-6f1dba72", intern = TRUE), 1))

cdr     # expect something like "project-id.C2024Q3R9" (exactly one dot)
bucket  # expect "gs://..."

Sys.setenv(WORKSPACE_CDR = cdr, WORKSPACE_BUCKET = bucket)

[1] "wb-silky-artichoke-2408.C2024Q3R9"

[1] "gs://rw-migration-aou-rw-6f1dba72"

In [2]:
BASE <- "/home/jupyter/workspace/rw-migration-aou-rw-6f1dba72/workspaces"
MISS <- file.path(BASE, "temporaldigitalhealthphenotyping/final_missing")
source(file.path(MISS, "load_missingness.R")); load_missingness(MISS)

EXPORT_DATE <- "20261011"            # the folder date of the 16 exports

L1_DIR  <- file.path(BASE, "result/final_missing/L1")
OUT_DIR <- file.path(BASE, "result/final_missing/results")
FINAL_PID   <- file.path(BASE, "result/final/final_pid.txt")
CCI_FILE    <- file.path(BASE, "result/pre_final/pre_final_cci.txt")
OVERLAP_POLICY <- "exclude"

cfg <- make_config(list(export  = list(date = EXPORT_DATE),
                        overlap = list(policy = OVERLAP_POLICY)))

cat("bucket :", cfg$export$bucket, "\nowner  :", cfg$export$owner, "\n")
cat("CDR    :", cfg$export$cdr, "\nbilling:", cfg$export$billing, "\n")
stopifnot("final_pid.txt not found - fix FINAL_PID"   = file.exists(FINAL_PID),
          "CCI file not found - fix CCI_FILE"         = file.exists(CCI_FILE),
          "WORKSPACE_BUCKET / OWNER_EMAIL not set"    = nzchar(cfg$export$bucket) && nzchar(cfg$export$owner))

bucket : gs://rw-migration-aou-rw-6f1dba72 
owner  : zqiu@researchallofus.org 
CDR    : wb-silky-artichoke-2408.C2024Q3R9 
billing: wb-dazzling-artichoke-8134 


In [3]:
need <- c("data.table", "lme4", "ordinal", "survival", "sandwich", "ggplot2", "rstatix", "ggpubr", "bigrquery")
print(sapply(need, requireNamespace, quietly = TRUE))   # any FALSE: install.packages("name")
#install.packages("sandwich")

Registered S3 method overwritten by 'car':
  method           from
  na.action.merMod lme4



data.table       lme4    ordinal   survival   sandwich    ggplot2    rstatix 
      TRUE       TRUE       TRUE       TRUE       TRUE       TRUE       TRUE 
    ggpubr  bigrquery 
      TRUE       TRUE 


In [4]:
cfg_demo <- make_config(list(boot = list(B = 50L)))
inp_demo <- simulate_inputs(cfg_demo)
res_demo <- run_missingness_pipeline(inp_demo$day_tables, inp_demo$persons, cfg_demo,
                                     file.path(tempdir(), "demo"),
                                     stages = c("missingness", "temporal", "descriptive", "onset"),
                                     onset_roles = "primary")
print(res_demo$primary_test[, .(stream, or = round(or, 2), p_holm)])    # expect or about 1.6-1.7 for all three

21:48:41 | hr: filled 6358 absent zone minute/calorie value(s) with 0

21:48:41 | hr: 1 day(s) have total zone minutes > 1440

Warning message:
“sleep: selection rule not fully met (n<21 days: 0; <70% observed: 4; gap>14 days: 0). Results are labelled as given.”
21:50:10 | [run] onset / activity / activity|joint|k1|adj  (session 0.6 GB, free 21.4 GB)

21:50:14 | [run] onset_extra / activity / vif  (session 0.6 GB, free 21.4 GB)

21:50:14 | [run] onset_extra / activity / cohort_interaction  (session 0.6 GB, free 21.4 GB)

21:50:21 | [run] onset_extra / activity / cohort_Insomnia  (session 0.6 GB, free 21.0 GB)

21:50:22 | [run] onset_extra / activity / cohort_Diabetes  (session 0.6 GB, free 21.2 GB)

21:50:24 | [run] onset_extra / activity / cohort_Healthy  (session 0.6 GB, free 21.1 GB)

21:50:26 | [run] onset_extra / activity / cohort_Prostate  (session 0.6 GB, free 21.1 GB)

21:50:26 | [run] onset_extra / activity / risk_unadjusted  (session 0.6 GB, free 21.4 GB)

21:50:29 | [run] on

     stream    or       p_holm
     <char> <num>        <num>
1: activity  1.59 6.207585e-22
2:       hr  1.73 2.115434e-41
3:    sleep  1.70 2.227767e-51


In [5]:
final_pid <- read_final_pid(FINAL_PID)
stopifnot("cohort names in final_pid must be Insomnia / Diabetes / Healthy / Prostate" =
            all(final_pid$cohort %in% cfg$cohorts))
print(final_pid[, .N, by = cohort])
ov <- resolve_cohort_overlap(final_pid, OVERLAP_POLICY, cfg$overlap$priority)
cat("unique people:", ov$n_unique_people, "| in more than one cohort:", length(ov$overlap_ids),
    "| kept with policy", OVERLAP_POLICY, ":", ov$n_kept, "\n")
print(ov$overlap)                      

     cohort     N
     <char> <int>
1: Diabetes  1298
2: Insomnia  1383
3: Prostate   250
4:  Healthy  6381
unique people: 8926 | in more than one cohort: 383 | kept with policy exclude : 8543 
                      combination n_people
                           <char>    <int>
1:            Diabetes + Insomnia      294
2:            Diabetes + Prostate       44
3:            Insomnia + Prostate       42
4: Diabetes + Insomnia + Prostate        3


In [6]:
#Export from BigQuery test - test on PCA
run_bq_exports(cfg, final_pid, cohorts = "Prostate")

21:52:55 | export Prostate/activity: running query for 250 people

21:53:03 | export Prostate/sleep: running query for 250 people

21:53:09 | export Prostate/hr: running query for 250 people

21:53:15 | export Prostate/meta: running query for 250 people



     cohort   stream n_people   status      note
     <char>   <char>    <int>   <char>    <char>
1: Prostate activity      250 exported 1 file(s)
2: Prostate    sleep      250 exported 1 file(s)
3: Prostate       hr      250 exported 1 file(s)
4: Prostate     meta      250 exported 1 file(s)


In [13]:
exports <- run_bq_exports(cfg, final_pid)

21:57:41 | export Insomnia/activity: running query for 1383 people

21:57:51 | export Insomnia/sleep: running query for 1383 people

21:57:59 | export Insomnia/hr: running query for 1383 people

21:58:11 | export Insomnia/meta: running query for 1383 people

21:58:18 | export Diabetes/activity: running query for 1298 people

21:58:28 | export Diabetes/sleep: running query for 1298 people

21:58:36 | export Diabetes/hr: running query for 1298 people

21:58:49 | export Diabetes/meta: running query for 1298 people

21:58:54 | export Healthy/activity: running query for 6381 people

21:59:06 | export Healthy/sleep: running query for 6381 people

21:59:20 | export Healthy/hr: running query for 6381 people

21:59:56 | export Healthy/meta: running query for 6381 people

22:00:02 | export Prostate/activity: 1 file(s) already there - skipped

22:00:03 | export Prostate/sleep: 1 file(s) already there - skipped

22:00:05 | export Prostate/hr: 1 file(s) already there - skipped

22:00:06 | export Pr

      cohort   stream n_people   status               note
      <char>   <char>    <int>   <char>             <char>
 1: Insomnia activity     1383 exported          1 file(s)
 2: Insomnia    sleep     1383 exported          1 file(s)
 3: Insomnia       hr     1383 exported          1 file(s)
 4: Insomnia     meta     1383 exported          1 file(s)
 5: Diabetes activity     1298 exported          1 file(s)
 6: Diabetes    sleep     1298 exported          1 file(s)
 7: Diabetes       hr     1298 exported          1 file(s)
 8: Diabetes     meta     1298 exported          1 file(s)
 9:  Healthy activity     6381 exported          6 file(s)
10:  Healthy    sleep     6381 exported          1 file(s)
11:  Healthy       hr     6381 exported          1 file(s)
12:  Healthy     meta     6381 exported          1 file(s)
13: Prostate activity      250  skipped 1 existing file(s)
14: Prostate    sleep      250  skipped 1 existing file(s)
15: Prostate       hr      250  skipped 1 existing file(

In [14]:
for (co in cfg$cohorts) for (s in c("activity", "sleep", "hr", "meta")) {
  n <- tryCatch(length(gsutil_ls(export_path(cfg, co, s))), error = function(e) 0)
  cat(sprintf("%-9s %-9s %d file(s)\n", co, s, n))
}
# any "0 file(s)" = a missing export

Insomnia  activity  1 file(s)
Insomnia  sleep     1 file(s)
Insomnia  hr        1 file(s)
Insomnia  meta      1 file(s)
Diabetes  activity  1 file(s)
Diabetes  sleep     1 file(s)
Diabetes  hr        1 file(s)
Diabetes  meta      1 file(s)
Healthy   activity  6 file(s)
Healthy   sleep     1 file(s)
Healthy   hr        1 file(s)
Healthy   meta      1 file(s)
Prostate  activity  1 file(s)
Prostate  sleep     1 file(s)
Prostate  hr        1 file(s)
Prostate  meta      1 file(s)


In [15]:
#Build the compact tables (once per export
etl <- run_etl(cfg, L1_DIR,
               final_pid_path = file.path(BASE, "result/final/final_pid.txt"),
               cci_path       = file.path(BASE, "result/pre_final/pre_final_cci.txt"))
print(attr(etl$day_tables$activity, "dropped_columns"))
print(attr(etl$day_tables$sleep,    "dropped_columns"))
print(attr(etl$day_tables$hr,       "dropped_columns"))
print(etl$persons[, .N, by = cohort])  
print(etl$summary)                                   # rows / people / date range of each day table
print(etl$dropped_columns)                           # columns removed because they are mostly NA or constant
print(etl$persons[, .N, by = cohort])

22:01:07 | cohorts: 8926 people in final_pid, 383 belong to more than one cohort; policy 'exclude' keeps 8543



                      combination n_people
                           <char>    <int>
1:            Diabetes + Insomnia      294
2:            Diabetes + Prostate       44
3:            Insomnia + Prostate       42
4: Diabetes + Insomnia + Prostate        3


22:01:10 | reading 1 file(s) for insomnia_activity_*.csv

22:01:13 | reading 1 file(s) for diabetes_activity_*.csv

22:01:17 | reading 6 file(s) for healthy_activity_*.csv

22:01:34 | reading 1 file(s) for pca_activity_*.csv

22:01:59 | ETL activity: 4157233 rows, 8543 people, dropped columns: 

22:02:01 | reading 1 file(s) for insomnia_sleep.csv

22:02:04 | sleep: 5 day(s) with efficiency outside [0,100] set to NA

22:02:06 | reading 1 file(s) for diabetes_sleep.csv

22:02:08 | sleep: 1 day(s) with efficiency outside [0,100] set to NA

22:02:09 | reading 1 file(s) for healthy_sleep.csv

22:02:17 | sleep: 7 day(s) with efficiency outside [0,100] set to NA

22:02:19 | reading 1 file(s) for pca_sleep.csv

22:02:41 | ETL sleep: 3548818 rows, 8543 people, dropped columns: 

22:02:43 | reading 1 file(s) for insomnia_hr.csv

22:02:48 | hr: 1102 day(s) have total zone minutes > 1440

22:02:49 | reading 1 file(s) for diabetes_hr.csv

22:02:54 | hr: 848 day(s) have total zone minutes > 1440

22

NULL
NULL
NULL
     cohort     N
     <fctr> <int>
1:  Healthy  6381
2: Insomnia  1044
3: Diabetes   957
4: Prostate   161
     stream  n_rows n_persons n_columns
     <char>   <int>     <int>     <int>
1: activity 4157233      8543        14
2:    sleep 3548818      8543        16
3:       hr 3375496      8543        19
                                                                                                                                                                                                                                      columns
                                                                                                                                                                                                                                       <char>
1:                                        person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,st

In [20]:
#Check the heart-rate table, total_hr_min should be at most 1440
hr <- etl$day_tables$hr
hr_cols <- c("total_hr_min", "elevated_min", "mins_out_of_range", "mins_fat_burn",
                       "mins_cardio", "mins_peak", "total_cal")
print(summary(hr[, hr_cols]))

Length  Class   Mode 
     0   NULL   NULL 


In [23]:
#Load the compact tables
inp <- load_inputs(cfg, L1_DIR)                      # nothing big is read: each stream is read when a stage needs it
print(data.frame(file = basename(inp$day_tables$paths), MB = round(file.size(inp$day_tables$paths) / 1e6, 1)))
cat("people:", nrow(inp$persons), "\n")
cat(sprintf("this R session: %.1f GB | free: %.1f GB | a model may use up to: %.1f GB\n",
            rss_gb(), available_memory_gb(), memory_headroom_gb(cfg)))

              file   MB
1 day_activity.rds 64.0
2    day_sleep.rds 59.7
3       day_hr.rds 99.4
4      persons.rds  0.1
people: 8543 
this R session: 0.6 GB | free: 21.2 GB | a model may use up to: 17.2 GB


In [24]:
# a) quick part: cohort tables, temporal structure, descriptive statistics, figures
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR,
                                stages = c("missingness", "temporal", "descriptive"))
print(res$cohort_missingness)
print(res$selection_check)        # confirms the upstream filter (>=70% observed, gap <=14, >=21 days)

Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes.”
Warning message:
“Groups with fewer than two datapoints have been drop

      stream   cohort n_persons followup_days   record_days missing_pct
      <char>   <fctr>     <int>        <char>        <char>      <char>
 1: activity Insomnia      1044  113 (97-497)  112 (96-484)     0 (0-2)
 2: activity Diabetes       957  116 (99-474)  115 (97-472)     0 (0-2)
 3: activity  Healthy      6381 151 (102-741) 147 (101-736)     0 (0-2)
 4: activity Prostate       161  108 (95-442)  107 (93-425)     0 (0-1)
 5:       hr Insomnia      1044  111 (91-331)  102 (85-328)     3 (0-8)
 6:       hr Diabetes       957  112 (91-302)  104 (87-291)     3 (0-7)
 7:       hr  Healthy      6381  124 (91-635)  121 (88-617)     2 (0-6)
 8:       hr Prostate       161  107 (86-299)   96 (81-293)     3 (0-6)
 9:    sleep Insomnia      1044  108 (90-418)   95 (78-393)    7 (3-14)
10:    sleep Diabetes       957  111 (91-365)   96 (80-323)    7 (3-15)
11:    sleep  Healthy      6381  121 (91-645)  108 (82-605)    6 (2-14)
12:    sleep Prostate       161  102 (83-378)   93 (71-329)    5

In [25]:
# b) the primary test and gap-definition models (the long part: probably around 3 hours)
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "onset",
                                onset_roles = c("primary", "secondary_threshold"),
                                model_streams = "activity")
print(res$primary_test)

22:18:22 | [run] onset / activity / activity|joint|k1|adj  (session 1.6 GB, free 20.1 GB)

22:22:50 | [run] onset / activity / activity|joint|k2|adj  (session 3.4 GB, free 18.3 GB)

22:28:08 | [run] onset / activity / activity|joint|k3|adj  (session 3.4 GB, free 18.3 GB)

22:33:22 | [run] onset / activity / activity|joint|k7|adj  (session 3.4 GB, free 18.4 GB)

22:39:34 | [run] onset_extra / activity / vif  (session 3.4 GB, free 18.3 GB)

22:39:35 | [run] onset_extra / activity / cohort_interaction  (session 3.4 GB, free 18.3 GB)

22:48:23 | [run] onset_extra / activity / cohort_Insomnia  (session 3.4 GB, free 18.4 GB)

22:48:53 | [run] onset_extra / activity / cohort_Diabetes  (session 3.4 GB, free 18.3 GB)

22:49:14 | [run] onset_extra / activity / cohort_Healthy  (session 3.4 GB, free 18.3 GB)

22:52:12 | [run] onset_extra / activity / cohort_Prostate  (session 3.4 GB, free 18.3 GB)



     stream        term   estimate       or   or_low  or_high     p p_holm
     <char>      <char>      <num>    <num>    <num>    <num> <num>  <num>
1: activity w_act_steps -0.4253611 1.530143 1.500349 1.560528     0      0
   n_events status
      <int> <char>
1:    30125     ok


In [26]:
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "onset",
                                onset_roles = c("primary", "secondary_threshold"),
                                model_streams = "sleep")
print(res$primary_test)

23:07:20 | [run] onset / sleep / sleep|joint|k1|adj  (session 1.6 GB, free 20.1 GB)

23:12:13 | [run] onset / sleep / sleep|joint|k2|adj  (session 3.6 GB, free 18.0 GB)

23:15:41 | [run] onset / sleep / sleep|joint|k3|adj  (session 3.6 GB, free 18.0 GB)

23:20:08 | [run] onset / sleep / sleep|joint|k7|adj  (session 3.6 GB, free 18.1 GB)

23:24:29 | [run] onset_extra / sleep / vif  (session 3.6 GB, free 18.0 GB)

23:24:30 | [run] onset_extra / sleep / cohort_interaction  (session 3.6 GB, free 18.0 GB)

23:34:38 | [run] onset_extra / sleep / cohort_Insomnia  (session 3.6 GB, free 18.0 GB)

23:35:29 | [run] onset_extra / sleep / cohort_Healthy  (session 3.6 GB, free 18.0 GB)

23:38:34 | [run] onset_extra / sleep / cohort_Prostate  (session 3.6 GB, free 18.0 GB)

23:38:40 | [run] onset_extra / sleep / risk_unadjusted  (session 3.6 GB, free 18.0 GB)



     stream        term   estimate       or   or_low  or_high     p p_holm
     <char>      <char>      <num>    <num>    <num>    <num> <num>  <num>
1: activity w_act_steps -0.4253611 1.530143 1.500349 1.560528     0      0
2:    sleep w_act_steps -0.2920882 1.339221 1.320109 1.358610     0      0
   n_events status
      <int> <char>
1:    30125     ok
2:   130895     ok


In [27]:
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "onset",
                                onset_roles = c("primary", "secondary_threshold"),
                                model_streams = "hr")
print(res$primary_test)

23:40:12 | [run] onset / hr / hr|joint|k1|adj  (session 1.6 GB, free 19.7 GB)

23:43:28 | [run] onset / hr / hr|joint|k2|adj  (session 3.8 GB, free 17.5 GB)

23:47:27 | [run] onset / hr / hr|joint|k3|adj  (session 3.8 GB, free 17.5 GB)

23:51:18 | [run] onset / hr / hr|joint|k7|adj  (session 3.8 GB, free 17.5 GB)

23:55:03 | [run] onset_extra / hr / vif  (session 3.8 GB, free 17.5 GB)

23:55:04 | [run] onset_extra / hr / cohort_interaction  (session 3.8 GB, free 17.5 GB)

00:01:34 | [run] onset_extra / hr / cohort_Insomnia  (session 3.8 GB, free 17.6 GB)

00:01:57 | [run] onset_extra / hr / cohort_Diabetes  (session 3.8 GB, free 17.6 GB)

00:02:16 | [run] onset_extra / hr / cohort_Healthy  (session 3.8 GB, free 17.5 GB)

00:04:41 | [run] onset_extra / hr / cohort_Prostate  (session 3.8 GB, free 17.5 GB)

00:04:46 | [run] onset_extra / hr / risk_unadjusted  (session 3.8 GB, free 17.5 GB)



     stream        term   estimate       or   or_low  or_high            p
     <char>      <char>      <num>    <num>    <num>    <num>        <num>
1: activity w_act_steps -0.4253611 1.530143 1.500349 1.560528 0.000000e+00
2:       hr w_act_steps -0.2341493 1.263833 1.236713 1.291548 2.392945e-99
3:    sleep w_act_steps -0.2920882 1.339221 1.320109 1.358610 0.000000e+00
         p_holm n_events status
          <num>    <int> <char>
1: 0.000000e+00    30125     ok
2: 2.392945e-99    47167     ok
3: 0.000000e+00   130895     ok


In [28]:
# Models B and C: 6 independent units per stream (4 x B, 2 x C), each saved on its own. They are built from
# the gaps and one activity predictor only (no person-day table), so they need little memory.
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "gap_models",
                                model_streams = c("activity", "sleep", "hr"))
print(res$model_B_post_gap_activity[, .(stream, model_id, estimate = round(estimate, 3), p = signif(p, 3), status)])
print(res$model_C_gap_length[, .(stream, model_id, or = round(or, 2), po_p = signif(po_p, 2), status)])
cat("failed units:", nrow(res$unit_failures), "\n")      # 0 = nothing failed; otherwise run this cell again to retry them
if (nrow(res$unit_failures) > 0L) print(res$unit_failures)

  <char>          <char>    <num>    <num> <char>
 1: activity     B_back1_adj   -0.079 6.36e-61     ok
 2: activity     B_back2_adj   -0.009 2.67e-02     ok
 3: activity B_back2_adj_pre    0.004 2.77e-01     ok
 4: activity   B_back2_unadj   -0.008 4.04e-02     ok
 5:       hr     B_back1_adj   -0.014 1.19e-09     ok
 6:       hr     B_back2_adj   -0.002 3.16e-01     ok
 7:       hr B_back2_adj_pre    0.001 6.99e-01     ok
 8:       hr   B_back2_unadj   -0.002 3.12e-01     ok
 9:    sleep     B_back1_adj   -0.017 2.57e-23     ok
10:    sleep     B_back2_adj   -0.020 4.49e-21     ok
11:    sleep B_back2_adj_pre   -0.012 9.12e-09     ok
12:    sleep   B_back2_unadj   -0.020 1.94e-21     ok
     stream model_id    or    po_p      status
     <char>   <char> <num>   <num>      <char>
1: activity    C_adj    NA 5.7e-20 po_rejected
2: activity  C_unadj    NA 5.9e-20 po_rejected
3:       hr    C_adj    NA 4.6e-18 po_rejected
4:       hr  C_unadj    NA 4.8e-18 po_rejected
5:    sleep    C_adj

In [29]:
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg, OUT_DIR, stages = "sensitivity",
                                model_streams = c("activity", "sleep", "hr"))
print(res$sensitivity[, .(stream, analysis, term, or = round(or, 2), p = signif(p, 3), status)])

03:29:12 | [skip] onset / activity / activity|joint|k1|adj (checkpoint from 2026-10-11 22:22)

03:29:12 | [run] sensitivity / activity / main  (session 1.6 GB, free 19.8 GB)

03:29:12 | [run] sensitivity / activity / trailing_baseline  (session 1.6 GB, free 19.8 GB)

03:33:52 | [run] sensitivity / activity / exclude_partial_wear  (session 3.4 GB, free 17.9 GB)

03:37:29 | [run] sensitivity / activity / first_gap_only  (session 3.4 GB, free 17.9 GB)

03:41:01 | [run] sensitivity / activity / conditional_logit  (session 3.4 GB, free 18.0 GB)

03:41:27 | [run] sensitivity / activity / cluster_robust_logit  (session 3.4 GB, free 18.0 GB)

03:43:28 | [run] sensitivity / activity / spline_vs_linear  (session 3.4 GB, free 18.0 GB)

03:51:50 | [run] sensitivity / activity / random_slope  (session 3.4 GB, free 18.0 GB)

04:07:23 | [skip] onset / sleep / sleep|joint|k1|adj (checkpoint from 2026-10-11 23:12)

04:07:23 | [run] sensitivity / sleep / main  (session 2.0 GB, free 19.4 GB)

04:07:23 | 

      stream             analysis         term    or         p status
      <char>               <char>       <char> <num>     <num> <char>
 1: activity cluster_robust_logit  w_act_steps  1.42 1.85e-154     ok
 2: activity    conditional_logit  w_act_steps  1.61  0.00e+00     ok
 3: activity exclude_partial_wear  w_act_steps  1.65  5.80e-98     ok
 4: activity       first_gap_only  w_act_steps  1.13  8.66e-12     ok
 5: activity                 main  w_act_steps  1.53  0.00e+00     ok
 6: activity         random_slope  w_act_steps    NA 2.65e-125     ok
 7: activity     spline_vs_linear  w_act_steps    NA  6.45e-51     ok
 8: activity    trailing_baseline tw_act_steps  1.78  0.00e+00     ok
 9:       hr cluster_robust_logit  w_act_steps  1.22  1.66e-47     ok
10:       hr    conditional_logit  w_act_steps  1.25  3.87e-77     ok
11:       hr exclude_partial_wear  w_act_steps  1.41  1.26e-67     ok
12:       hr       first_gap_only  w_act_steps  1.26  2.87e-22     ok
13:       hr        

In [30]:
cfg_fast <- make_config(list(export  = list(date = EXPORT_DATE),
                             overlap = list(policy = OVERLAP_POLICY),
                             model   = list(exploratory_sample_persons = 3000L)))
res <- run_missingness_pipeline(inp$day_tables, inp$persons, cfg_fast, OUT_DIR, stages = "onset",
                                onset_roles = "exploratory", model_streams = c("activity", "sleep", "hr"))
print(table(res$onset_models$role, res$onset_models$stream))

05:25:56 | [run] onset / activity / activity|uni_act_steps|k1|unadj  (session 2.2 GB, free 19.1 GB)

05:27:01 | [run] onset / activity / activity|uni_act_steps|k1|adj  (session 3.2 GB, free 18.1 GB)

05:28:20 | [run] onset / activity / activity|uni_act_sedentary|k1|unadj  (session 3.2 GB, free 18.0 GB)

05:29:15 | [run] onset / activity / activity|uni_act_sedentary|k1|adj  (session 3.2 GB, free 18.1 GB)

05:30:27 | [run] onset / activity / activity|uni_act_mvpa|k1|unadj  (session 3.2 GB, free 18.1 GB)

05:31:23 | [run] onset / activity / activity|uni_act_mvpa|k1|adj  (session 3.2 GB, free 18.1 GB)

05:32:32 | [run] onset / activity / activity|joint|k1|unadj  (session 3.2 GB, free 18.1 GB)

05:33:49 | [skip] onset_extra / activity / vif (checkpoint from 2026-10-11 22:39)

05:33:49 | [run] onset / sleep / sleep|own_uni_sleep_inbed|k1|unadj  (session 2.6 GB, free 18.7 GB)

05:34:23 | [run] onset / sleep / sleep|own_uni_sleep_inbed|k1|adj  (session 4.0 GB, free 17.3 GB)

05:35:43 | [run] o

                     
                      activity hr sleep
  exploratory                7 15    15
  primary                    1  1     1
  secondary_threshold        3  3     3


In [31]:
CKPT <- file.path(OUT_DIR, cfg$checkpoint$dirname)
print(checkpoint_report(CKPT, cfg)[, .N, by = .(kind, state)])     # valid / stale / failed / corrupt checkpoints
cat(readLines(file.path(OUT_DIR, "results_summary.md")), sep = "\n")
print(list.files(file.path(OUT_DIR, "tables")))
print(list.files(file.path(OUT_DIR, "figures")))                   # figure_captions.md explains every figure
system2("gsutil", c("-m", "cp", "-r", OUT_DIR, file.path(Sys.getenv("WORKSPACE_BUCKET"), "missingness_results/")))

          kind   state     N
        <char>  <char> <int>
1:        <NA> corrupt     9
2:        gapB   valid    12
3:        gapC   valid     6
4:       onset   valid    49
5: onset_extra   valid    21
6: sensitivity   valid    25
# Results summary

Written from the saved result tables (no model is refitted).

## Gap onset (primary model, gap >= 1 day)
Odds that a gap starts tomorrow per 1 SD below the person's usual steps yesterday (OR > 1 = lower activity before gaps).
* activity: OR 1.53 (1.50-1.56), p=0, Holm p=0 [ok,  30,125 events]
* hr: OR 1.26 (1.24-1.29), p=2.39e-99, Holm p=2.39e-99 [ok,  47,167 events]
* sleep: OR 1.34 (1.32-1.36), p=0, Holm p=0 [ok, 130,895 events]

 [1] "calendar_year_rates.tsv"         "checkpoint_status.tsv"          
 [3] "cohort_interaction.tsv"          "cohort_missingness.tsv"         
 [5] "cohort_specific.tsv"             "cohort_tests_overall.tsv"       
 [7] "cohort_tests_pairwise.tsv"       "gap_runs.tsv"                   
 [9] "metric_summary_